In [ ]:
# ========================================
# IMPORTS AND SETUP
# ========================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
import json
import joblib
import time
import gc
import os
import subprocess
from datetime import datetime

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import xgboost as xgb

print(f"Notebook run: {datetime.now().isoformat()}")
print(f"XGBoost version: {xgb.__version__}")

# ========================================
# GPU DETECTION
# ========================================
def detect_gpu():
    """Detect if CUDA GPU is available for XGBoost"""
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            print("✓ NVIDIA GPU detected!")
            print(result.stdout.split('\n')[0])  # First line of nvidia-smi
            return True
    except Exception:
        pass
    print("⚠ No GPU detected - using CPU (slower)")
    return False

GPU_AVAILABLE = detect_gpu()

# XGBoost device configuration
if GPU_AVAILABLE:
    DEVICE = 'cuda'
    TREE_METHOD = 'hist'
    N_JOBS = 1  # GPU uses 1 job
    print(f"Using GPU acceleration: device='{DEVICE}', tree_method='{TREE_METHOD}'")
else:
    DEVICE = 'cpu'
    TREE_METHOD = 'hist'
    N_JOBS = -1  # CPU uses all cores
    print(f"Using CPU: device='{DEVICE}', tree_method='{TREE_METHOD}', n_jobs={N_JOBS}")

In [ ]:
# ========================================
# LOAD DATA
# ========================================
import re

# Flexible path detection for both local and Amarel
if Path.cwd().name == 'notebooks_clean':
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

DATA_DIR = ROOT / 'data'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

print(f"ROOT: {ROOT}")
print(f"DATA_DIR: {DATA_DIR}")

X_train = pd.read_csv(DATA_DIR / 'X_train.csv')
X_test = pd.read_csv(DATA_DIR / 'X_test.csv')
y_train = pd.read_csv(DATA_DIR / 'y_train.csv')['ClosePrice'].values
y_test = pd.read_csv(DATA_DIR / 'y_test.csv')['ClosePrice'].values

# Sanitize column names for XGBoost
def _clean_col(c):
    return re.sub(r'[^0-9a-zA-Z_]', '_', str(c))

orig_cols = list(X_train.columns)
new_cols = [_clean_col(c) for c in orig_cols]
if new_cols != orig_cols:
    print('Sanitizing feature names for XGBoost...')
    X_train.columns = new_cols
    X_test.columns = [_clean_col(c) for c in X_test.columns]

print(f"\nTraining: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
print(f"Testing: {X_test.shape[0]:,} samples")

In [ ]:
# ========================================
# EVALUATION FUNCTION
# ========================================
STEPH_THRESHOLD = 0.884  # Steph's benchmark
TARGET_THRESHOLD = 0.85  # Our minimum target

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Comprehensive model evaluation with status indicators"""
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    results = {
        'model': model_name,
        'train_r2': r2_score(y_train, y_pred_train),
        'test_r2': r2_score(y_test, y_pred_test),
        'test_rmse': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'test_mae': mean_absolute_error(y_test, y_pred_test),
        'test_mdape': np.median(np.abs((y_test - y_pred_test) / y_test)) * 100,
    }

    print(f"\n{'='*70}")
    print(f"{model_name}")
    print(f"{'='*70}")
    print(f"Train R²: {results['train_r2']:.4f}")
    
    if results['test_r2'] > STEPH_THRESHOLD:
        status = '🎉 BEATS STEPH!'
    elif results['test_r2'] >= TARGET_THRESHOLD:
        status = '✓ TARGET ACHIEVED (85%+)'
    else:
        gap = (STEPH_THRESHOLD - results['test_r2']) * 100
        status = f"(Gap: {gap:.2f}% to Steph)"
    
    print(f"Test R²:  {results['test_r2']:.4f} {status}")
    print(f"RMSE:     ${results['test_rmse']:,.0f}")
    print(f"MAE:      ${results['test_mae']:,.0f}")
    print(f"MdAPE:    {results['test_mdape']:.2f}%")
    
    return results, model

print("✓ Setup complete. Ready to train XGBoost.")

In [ ]:
# ========================================
# STAGE 1: BROAD PARAMETER SEARCH
# ========================================
print("XGBoost GPU-Accelerated Tuning")
print('='*70)
print(f"Target: >{TARGET_THRESHOLD*100:.0f}% R² | Benchmark: {STEPH_THRESHOLD*100:.1f}% R²")
print(f"Device: {DEVICE.upper()}")
print('='*70)

# Split off validation set for early stopping
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)
print(f"\nTrain split: {X_tr.shape[0]:,} samples")
print(f"Val split:   {X_val.shape[0]:,} samples")

# Stage 1: Broad Search
print('\n' + '='*70)
print("STAGE 1: Broad Parameter Search (20 iterations, CV=3)")
print('='*70)

stage1_params = {
    'n_estimators': [300, 500, 700, 1000],
    'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.07],
    'max_depth': [6, 8, 10, 12, 15],
    'min_child_weight': [1, 3, 5, 7],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.2, 0.3],
    'reg_alpha': [0, 0.1, 0.5, 1.0],
    'reg_lambda': [0.5, 1.0, 2.0, 5.0],
}

stage1_start = time.time()

# Base estimator with GPU/CPU settings
base_estimator = xgb.XGBRegressor(
    random_state=42,
    device=DEVICE,
    tree_method=TREE_METHOD,
    n_jobs=N_JOBS,
    verbosity=0
)

stage1_search = RandomizedSearchCV(
    base_estimator,
    param_distributions=stage1_params,
    n_iter=20,
    cv=3,
    scoring='r2',
    random_state=42,
    verbose=1,
    n_jobs=2 if GPU_AVAILABLE else 4  # Limit parallel CV on GPU
)

stage1_search.fit(X_tr, y_tr)
stage1_time = time.time() - stage1_start

print(f"\nStage 1 Complete ({stage1_time:.1f}s)")
print(f"Best CV R²: {stage1_search.best_score_:.4f}")
print(f"Best Params: {stage1_search.best_params_}")

In [ ]:
# ========================================
# STAGE 2: REFINED SEARCH
# ========================================
print('='*70)
print("STAGE 2: Refined Search Around Best Parameters (25 iterations, CV=3)")
print('='*70)

bp = stage1_search.best_params_

# Create refined ranges around best params
stage2_params = {
    'n_estimators': [max(200, bp['n_estimators']-200), bp['n_estimators'], bp['n_estimators']+200, bp['n_estimators']+400],
    'learning_rate': [max(0.005, bp['learning_rate']*0.7), bp['learning_rate'], min(0.1, bp['learning_rate']*1.3)],
    'max_depth': [max(4, bp['max_depth']-2), bp['max_depth'], min(20, bp['max_depth']+2)],
    'min_child_weight': [max(1, bp['min_child_weight']-2), bp['min_child_weight'], bp['min_child_weight']+2],
    'subsample': [max(0.6, bp['subsample']-0.1), bp['subsample'], min(1.0, bp['subsample']+0.05)],
    'colsample_bytree': [max(0.6, bp['colsample_bytree']-0.1), bp['colsample_bytree'], min(1.0, bp['colsample_bytree']+0.05)],
    'gamma': [max(0, bp['gamma']-0.1), bp['gamma'], bp['gamma']+0.1],
    'reg_alpha': [max(0, bp['reg_alpha']*0.5), bp['reg_alpha'], bp['reg_alpha']*1.5] if bp['reg_alpha'] > 0 else [0, 0.05, 0.1],
    'reg_lambda': [max(0.1, bp['reg_lambda']*0.5), bp['reg_lambda'], bp['reg_lambda']*1.5],
}

stage2_start = time.time()

stage2_search = RandomizedSearchCV(
    xgb.XGBRegressor(
        random_state=42,
        device=DEVICE,
        tree_method=TREE_METHOD,
        n_jobs=N_JOBS,
        verbosity=0
    ),
    param_distributions=stage2_params,
    n_iter=25,
    cv=3,
    scoring='r2',
    random_state=43,
    verbose=1,
    n_jobs=2 if GPU_AVAILABLE else 4
)

stage2_search.fit(X_tr, y_tr)
stage2_time = time.time() - stage2_start

print(f"\nStage 2 Complete ({stage2_time:.1f}s)")
print(f"Best CV R²: {stage2_search.best_score_:.4f}")
print(f"Best Params: {stage2_search.best_params_}")

In [ ]:
# ========================================
# STAGE 3: FINAL TRAINING WITH EARLY STOPPING
# ========================================
print('='*70)
print("STAGE 3: Final Training with Extended Estimators & Early Stopping")
print('='*70)

final_params = stage2_search.best_params_.copy()

# Increase estimators for final model (early stopping will prevent overfitting)
final_params['n_estimators'] = min(2000, final_params['n_estimators'] * 2)

print(f"Training with up to {final_params['n_estimators']} estimators...")
print(f"Early stopping will select optimal number.")

stage3_start = time.time()

xgb_model = xgb.XGBRegressor(
    **final_params,
    random_state=42,
    device=DEVICE,
    tree_method=TREE_METHOD,
    n_jobs=N_JOBS,
    early_stopping_rounds=50,
    verbosity=0
)

# Train with early stopping
xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

stage3_time = time.time() - stage3_start

# Get actual number of trees used
best_iter = getattr(xgb_model, 'best_iteration', final_params['n_estimators'])
if best_iter is None:
    best_iter = final_params['n_estimators']

print(f"\nStage 3 Complete ({stage3_time:.1f}s)")
print(f"Final model: {best_iter} trees (early stopped from {final_params['n_estimators']})")

# Total time
total_time = stage1_time + stage2_time + stage3_time
print(f"\nTotal tuning time: {total_time:.1f}s ({total_time/60:.1f} minutes)")

In [ ]:
# ========================================
# EVALUATE ON TEST SET
# ========================================
xgb_results, xgb_model = evaluate_model(xgb_model, X_train, X_test, y_train, y_test, "XGBoost (GPU-Tuned)")

# Save model and results
joblib.dump(xgb_model, MODELS_DIR / 'xgboost_gpu_tuned.joblib')
print(f"\n✓ Model saved to {MODELS_DIR / 'xgboost_gpu_tuned.joblib'}")

# Save parameters and metrics
save_params = final_params.copy()
save_params['actual_n_estimators'] = int(best_iter)
save_params['stage1_best_r2'] = float(stage1_search.best_score_)
save_params['stage2_best_r2'] = float(stage2_search.best_score_)
save_params['test_r2'] = float(xgb_results['test_r2'])
save_params['test_rmse'] = float(xgb_results['test_rmse'])
save_params['test_mae'] = float(xgb_results['test_mae'])
save_params['device_used'] = DEVICE
save_params['total_tuning_time_seconds'] = total_time

# Convert numpy types for JSON
for k, v in save_params.items():
    if hasattr(v, 'item'):
        save_params[k] = v.item()

with open(MODELS_DIR / 'xgboost_gpu_tuned_params.json', 'w') as f:
    json.dump(save_params, f, indent=2)

print(f"✓ Parameters saved to {MODELS_DIR / 'xgboost_gpu_tuned_params.json'}")

# Cleanup
del stage1_search, stage2_search
gc.collect()

In [ ]:
# ========================================
# FEATURE IMPORTANCE
# ========================================
print('\nTOP 20 MOST IMPORTANT FEATURES')
print('='*80)

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.head(20).to_string(index=False))

# Save feature importance
feature_importance.to_csv(MODELS_DIR / 'xgboost_gpu_feature_importance.csv', index=False)
print(f"\n✓ Feature importance saved to {MODELS_DIR / 'xgboost_gpu_feature_importance.csv'}")

In [ ]:
# ========================================
# ALSO SAVE AS BEST ADVANCED MODEL (for notebook 05)
# ========================================
# Save as best_advanced_model.joblib for compatibility with 05_model_analysis
joblib.dump(xgb_model, MODELS_DIR / 'best_advanced_model.joblib')

summary = {
    'best_model': 'XGBoost (GPU-Tuned)',
    'best_r2': float(xgb_results['test_r2']),
    'best_rmse': float(xgb_results['test_rmse']),
    'best_mae': float(xgb_results['test_mae']),
    'timestamp': datetime.now().isoformat(),
    'device': DEVICE,
    'tuning_time_seconds': total_time
}

with open(MODELS_DIR / 'advanced_models_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n{'='*70}")
print("FINAL SUMMARY")
print(f"{'='*70}")
print(f"Best Model: XGBoost (GPU-Tuned)")
print(f"Test R²: {xgb_results['test_r2']:.4f}")
print(f"RMSE: ${xgb_results['test_rmse']:,.0f}")
print(f"MAE: ${xgb_results['test_mae']:,.0f}")
print(f"Device: {DEVICE.upper()}")
print(f"Tuning Time: {total_time/60:.1f} minutes")
print(f"\n✓ All outputs saved to {MODELS_DIR}")